# 模型对比

先跑 `say` 合成的固定测试集（zh / en / ja / 日英混说 / 中英混说），再拿自己的录音对比。

- 首次运行会从 Hugging Face 下载模型（合计约 6.5 GB），之后全部本地推理
- `x_realtime` = 音频秒数 ÷ 推理秒数，越高越快；`WER/CER` 越低越准（英文用 WER，中日混说用去空格 CER）
- 每个 cell 加载模型后常驻内存，第二次调用起才是纯推理耗时

In [1]:
from asr_lab import ensure_testset

for clip in ensure_testset():
    print(f"{clip['name']:<14} [{clip['lang']:<5}] {clip['text']}")

zh_01          [zh   ] 今天下午三点，我们在会议室讨论新版本的功能设计，然后把结果同步给测试团队。
en_01          [en   ] Let's ship the new release tomorrow morning, and sync the results with the QA team.
ja_01          [ja   ] 明日の午後三時に、会議室で新機能について話し合います。
ja_en_mix_01   [mixed] このissueは次のsprintでfixする予定なので、まずquickなreviewをお願いします。
zh_en_mix_01   [mixed] 这个需求先做一个quick prototype，然后我们再review一下design。


In [2]:
import pandas as pd
from asr_lab import run_benchmark

pd.set_option("display.max_colwidth", 48)
df = run_benchmark()
df

/Users/fang2hou/Developer/projects/dito/asr-lab/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 1101.30it/s]

[warmup] sensevoice-small-int8 · sherpa-onnx CPU ready


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 2853.27it/s]

... (4 outputs left out)

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 3844.11it/s]

[warmup] qwen3-asr-1.7b · MLX ready


,clip,lang,model,detected,CER,elapsed_s,x_realtime,text,WER
0,zh_01,zh,sensevoice-small-int8 · sherpa-onnx CPU,zh,0.027,0.084,98.3,今天下午3点，我们在会议室讨论新版本的功能设计，然后把结果同步给测试团队。,NaN
1,en_01,en,sensevoice-small-int8 · sherpa-onnx CPU,en,NaN,0.050,90.4,Let's ship the new release tomorrow morning ...,0.067
2,ja_01,ja,sensevoice-small-int8 · sherpa-onnx CPU,ja,0.074,0.055,89.9,明日の午後3時に 会議室 で 新機能 について話し合います。,NaN
3,ja_en_mix_01,mixed,sensevoice-small-int8 · sherpa-onnx CPU,ja,0.540,0.070,97.0,この 異周 は 次 の スプリント で フィックス する 予定 な の で、まず クイッ...,NaN
4,zh_en_mix_01,mixed,sensevoice-small-int8 · sherpa-onnx CPU,zh,0.000,0.062,94.4,这个需求先做一个quick prototype，然后我们再review一下design。,NaN
5,zh_01,zh,whisper-large-v3-turbo · MLX,zh,0.081,1.008,8.2,"今天下午3点,我们在会议室讨论新版本的功能设计,然后把结果同步给测试团队。",NaN
6,en_01,en,whisper-large-v3-turbo · MLX,en,NaN,0.949,4.8,"Let's ship the new release tomorrow morning,...",0.000
7,ja_01,ja,whisper-large-v3-turbo · MLX,ja,0.037,0.971,5.1,明日の午後3時に、会議室で新機能について話し合います。,NaN
8,ja_en_mix_01,mixed,whisper-large-v3-turbo · MLX,ja,0.560,0.994,6.8,このイシューは、次のスプリントでフィックスする予定なので、まずクイックなレビューをお願い...,NaN
9,zh_en_mix_01,mixed,whisper-large-v3-turbo · MLX,zh,0.116,0.963,6.1,"这个需求先做一个Quick Prototype,然后我们再Review一下Design。",NaN


## 热词（专业词表）

Qwen3-ASR 原生支持解码级热词偏置：`context` 传空格分隔词表（注入 system prompt，等价官方 API 的 hotword 机制，流式同样支持）。Whisper 只能用 `initial_prompt` 做软偏置；SenseVoice 无此机制。

In [3]:
import subprocess

from asr_lab.models import load_model
from asr_lab.paths import AUDIO_DIR

text = "岐伯山的桫椤要移栽到蕲春，顺便核对美卓乐和芪苈强心胶囊的处方。"
wav = AUDIO_DIR / "hotword_zh.wav"
if not wav.exists():
    subprocess.run(["say", "-v", "Tingting", "-o", str(wav), "--data-format=LEI16@16000", text], check=True)

qwen = load_model("qwen3-asr-1.7b")
hotwords = "桫椤 蕲春 岐伯 美卓乐 芪苈强心胶囊"
print("无热词:", qwen.transcribe(wav).text)
print("有热词:", qwen.transcribe(wav, context=hotwords).text)
print("参考 :", text)

无热词: 齐博山的梭罗要移栽到齐春，顺便核对美卓乐和齐力强心胶囊的处方。


有热词: 岐伯山的桫椤要移栽到蕲春，顺便核对美卓乐和芪苈强心胶囊的处方。
参考 : 岐伯山的桫椤要移栽到蕲春，顺便核对美卓乐和芪苈强心胶囊的处方。


## 用自己的录音对比

默认取 `recordings/` 里最新的一条（没有则回退到日语文本）；也可以直接改 `audio` 为任意音频路径。想测指定语言提示，把 `language="Japanese"` 之类传给 `qwen3-asr` 的 `transcribe`。

In [4]:
from asr_lab import compare, latest_recording
from asr_lab.paths import AUDIO_DIR

audio = latest_recording() or AUDIO_DIR / "ja_en_mix_01.wav"
print("audio:", audio)
compare(audio)

audio: /Users/fang2hou/Developer/projects/dito/asr-lab/audio/ja_en_mix_01.wav


,model,detected,elapsed_s,x_realtime,text
0,sensevoice-small-int8 · sherpa-onnx CPU,ja,0.090,75.4,この 異周 は 次 の スプリント で フィックス する 予定 な の で、まず クイッ...
1,whisper-large-v3-turbo · MLX,ja,1.010,6.7,このイシューは、次のスプリントでフィックスする予定なので、まずクイックなレビューをお願い...
2,qwen3-asr-0.6b · MLX,Japanese,0.268,25.4,この一周は次のスプリントでフィックスする予定なので、まずクイックなレビューをお願いします。
3,qwen3-asr-1.7b · MLX,Japanese,0.621,11.0,このイシュは次のスプリントでフィックスする予定なので、まずクイックなレビューをお願いします。
